# 02. Comprensión y auditoría de datos

Esta notebook construye el universo analítico del proyecto de informalidad laboral con ENOE 1T 2026. Se cargan las tablas principales, se aplican filtros de población, se valida la cobertura entre SDEM, COE1 y COE2, se define la variable objetivo `EMP_PPAL`, se eliminan variables con fuga de información y se genera una base inicial para la selección posterior de predictores operativos.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dbfread import DBF

### Cargamos las bases sdem, coe1 y coe2

In [2]:
dbf_sdem = DBF("/home/jeffrey/Documentos/Proyectos/enoe_informalidad/data/raw/ENOE_SDEMT126.dbf")
dbf_coe1 = DBF("/home/jeffrey/Documentos/Proyectos/enoe_informalidad/data/raw/ENOE_COE1T126.dbf")
dbf_coe2 = DBF("/home/jeffrey/Documentos/Proyectos/enoe_informalidad/data/raw/ENOE_COE2T126.dbf")

df_sdem = pd.DataFrame(iter(dbf_sdem))
df_coe1 = pd.DataFrame(iter(dbf_coe1))
df_coe2 = pd.DataFrame(iter(dbf_coe2))

El filtro para la población ocupada que menciona la documentación oficial `Conociendo la base de datos`, menciona el uso de las variables `EDA`, `C_RES` y `R_DEF`

Observamos el tipo y la frecuencia de valores en la tabla df_sdem para EDA y C_RES

In [3]:
print(df_sdem["EDA"].dtype)
dict(df_sdem["EDA"].value_counts())

object


{'': np.int64(9651),
 '15': np.int64(7260),
 '18': np.int64(7212),
 '16': np.int64(7013),
 '17': np.int64(6989),
 '13': np.int64(6958),
 '11': np.int64(6927),
 '14': np.int64(6779),
 '12': np.int64(6776),
 '25': np.int64(6668),
 '23': np.int64(6578),
 '19': np.int64(6557),
 '10': np.int64(6556),
 '20': np.int64(6538),
 '21': np.int64(6460),
 '22': np.int64(6390),
 '09': np.int64(6329),
 '30': np.int64(6249),
 '08': np.int64(6240),
 '24': np.int64(6093),
 '31': np.int64(6004),
 '26': np.int64(6002),
 '33': np.int64(5992),
 '43': np.int64(5919),
 '35': np.int64(5916),
 '36': np.int64(5869),
 '40': np.int64(5824),
 '50': np.int64(5814),
 '07': np.int64(5780),
 '27': np.int64(5755),
 '37': np.int64(5708),
 '38': np.int64(5696),
 '28': np.int64(5672),
 '34': np.int64(5646),
 '41': np.int64(5600),
 '32': np.int64(5596),
 '45': np.int64(5524),
 '39': np.int64(5484),
 '51': np.int64(5469),
 '06': np.int64(5447),
 '42': np.int64(5405),
 '29': np.int64(5394),
 '48': np.int64(5387),
 '53': np.int

In [4]:
print(df_sdem["C_RES"].dtype)
dict(df_sdem["C_RES"].value_counts())

object


{'1': np.int64(400611), '2': np.int64(9651), '3': np.int64(7175)}

Observamos que hay 9651 conteos con el valor de cadena vacia '' que coinciden con C_RES=='2'. Veamos la tabla cruzada cuando EDA==''

In [5]:
pd.crosstab(df_sdem["EDA"].eq(""),df_sdem["C_RES"])

C_RES,1,2,3
EDA,,,
False,400611,0,7175
True,0,9651,0


Confirmamos entonces que cuando C_RES==2, aparecen cadenas vacias en EDA.

Veamos si existen espacios ocultos en EDA y C_RES

In [6]:
for col in ("EDA", "C_RES"):
    originales = df_sdem[col].astype(str)
    sin_espacios = originales.str.strip()
    
    print(col, originales.ne(sin_espacios).sum()) #ne() significa "no es igual"

# Por tanto, no hay espacios ocultos en EDA ni C_RES

EDA 0
C_RES 0


La documentación muestra el significado de los valores 

97: 97 años y más 

98:  Edad no especificada para mayores (12 años y más)

99:  Edad no especificada para menores de (00 a 11 años)

Veamos cuántos registros con C_RES ={1,3} tienen codigos EDA = 98 o 99

In [7]:
print(len(df_sdem[(df_sdem["EDA"]=='98') & ((df_sdem["C_RES"]=="1") | (df_sdem["C_RES"]=='3'))]))
print(len(df_sdem[(df_sdem["EDA"]=='99') & ((df_sdem["C_RES"]=="1") | (df_sdem["C_RES"]=='3'))]))

176
49


Con lo anterior, se definen las reglas de exclusión. Se descartarán los registros con `C_RES = 2`, ya que corresponden a ausentes definitivos y no forman parte de la población elegible. Los valores de `EDA` entre `00` y `97` se convertirán a formato numérico y se conservarán únicamente las personas de 15 años o más. El valor `EDA = 97` se mantendrá numéricamente como 97, pero se interpretará como “97 años o más”. Se excluirán los registros con `EDA = 98`, porque no permiten verificar si la persona cumple el criterio de edad mínima, y aquellos con `EDA = 99`, porque corresponden a menores de 12 años. Las cadenas vacías de `EDA` se excluirán junto con los registros con `C_RES = 2`.


### Apliquemos estas restricciones a la tabla sdem

In [8]:
# Elimina C_RES = 2
lon_original = len(df_sdem)
df_sdem_filtrado = df_sdem[df_sdem["C_RES"] != '2'].copy()
lon_desp_cres = len(df_sdem_filtrado)

print("Registros originales: ", lon_original)
print("Eliminados por C_RES = 2: ", lon_original - lon_desp_cres)

# Convertir los valores de EDA a numéricos
df_sdem_filtrado["EDA"] = pd.to_numeric(df_sdem_filtrado["EDA"])

# Filtrar los valores fuera de 15 y 97
df_sdem_filtrado = df_sdem_filtrado[(df_sdem_filtrado["EDA"]>=15) & (df_sdem_filtrado["EDA"]<=97)]
lon_desp_eda = len(df_sdem_filtrado)
print("Eliminados después por EDA fuera de 15-97: ", lon_desp_cres - lon_desp_eda)

lon_sdem_final = len(df_sdem_filtrado)
print("Registros finales: ", lon_sdem_final)

Registros originales:  417437
Eliminados por C_RES = 2:  9651
Eliminados después por EDA fuera de 15-97:  83786
Registros finales:  324000


Ademásm, la documentación indica que R_DEF representa el resultado definitivo de la entrevista. Se identifican los códigos "00" para entrevistas completas y "15" para entrevistas suspendidas. Debido a que el análisis requiere información laboral completa, se conservaron únicamente los registros con R_DEF = "00", obteniendo 323,124 personas elegibles.

In [9]:
df_sdem_filtrado["R_DEF"].value_counts(dropna=False)

R_DEF
00    323124
15       876
Name: count, dtype: int64

In [10]:
df_sdem_filtrado = df_sdem_filtrado[df_sdem_filtrado["R_DEF"]=='00']
print("Registros luego de eliminar R_DEF=15:",len(df_sdem_filtrado))

Registros luego de eliminar R_DEF=15: 323124


## Llaves compatibles entre tablas

La documentación de la base de datos menciona que la tabla tiene llave compuesta dada por

In [11]:
llave = ["TIPO","MES_CAL","CD_A", "CVE_ENT", "CON", "V_SEL", "N_HOG", "H_MUD", "N_REN"]

Estudiemos:

+ Personas de SDEM con llave presente en COE1.
+ Personas de SDEM sin llave en COE1.
+ Personas de SDEM con llave presente en COE2.
+ Personas de SDEM sin llave en COE2.

In [12]:
# Personas de SDEM con llave presente en COE1. Primero hagamos un merge

cobertura_coe1 = df_sdem_filtrado[llave].merge(
    df_coe1[llave],
    on=llave,
    how="left",
    indicator=True,
    validate='one_to_one'
)
print(cobertura_coe1["_merge"].value_counts())

# Obtengamos cantidades y porcentajes
conteos = cobertura_coe1["_merge"].value_counts()

presentes = conteos.get("both",0)
ausentes = conteos.get("left_only",0)
total = len(cobertura_coe1)

print("presentes en COE1: ", presentes, presentes / total * 100)
print("ausentes en COE1: ", ausentes, ausentes / total * 100)

_merge
both          323124
left_only          0
right_only         0
Name: count, dtype: int64
presentes en COE1:  323124 100.0
ausentes en COE1:  0 0.0


El 100% presentes en SDEM lo están también en COE1. Veamos en COE2

In [13]:
# Personas en SDEM con llave presente en COE2
cobertura_coe2 = df_sdem_filtrado[llave].merge(
    df_coe2[llave],
    on=llave,
    how="left",
    indicator=True,
    validate='one_to_one'
)
print(cobertura_coe2["_merge"].value_counts())

#Veamos sus porcentajes
conteos = cobertura_coe2["_merge"].value_counts()
presentes = conteos.get("both",0)
ausentes = conteos.get("left_only",0)
total = len(cobertura_coe2)

print("Presentes en COE2:", presentes, presentes / total * 100)
print("Ausentes en COE2:", ausentes, ausentes / total * 100)

_merge
both          323124
left_only          0
right_only         0
Name: count, dtype: int64
Presentes en COE2: 323124 100.0
Ausentes en COE2: 0 0.0


Después del filtrado, la tabla SDEM contiene 323,124 registros. Todas sus llaves están presentes en `COE1` y `COE2`, por lo que la cobertura de la población elegible es del 100 % en ambas tablas.

Asimismo, no existen llaves duplicadas, por lo que la relación entre las tres tablas es uno a uno para esta población. En consecuencia, una unión posterior mediante la llave compuesta no debería eliminar ni multiplicar registros, siempre que se utilice esta población depurada y se valide la cardinalidad del cruce.

Encontremos entonces lo siguiente luego de los filtrados

Para cada tabla,
+ número de filas y columnas;
+ columnas compartidas entre COE1 y COE2;
+ columnas exclusivas de cada tabla;
+ columnas adicionales a la llave.

Filtremos primero COE1 y COE2

In [14]:
coe1_filtrado = df_coe1.merge(
    df_sdem_filtrado[llave].drop_duplicates(),
    on=llave,
    how='inner'
    )

coe2_filtrado = df_coe2.merge(
    df_sdem_filtrado[llave].drop_duplicates(),
    on=llave,
    how='inner'
)

In [15]:
print("Número de registros para la tabla df_sdem_filtrado:", df_sdem_filtrado.shape[0], "Numero de columnas:", df_sdem_filtrado.shape[1])
print("Número de registros para la tabla coe1_filtrado:", coe1_filtrado.shape[0], "Numero de columnas:",coe1_filtrado.shape[1])
print("Número de registros para la tabla coe2_filtrado:", coe2_filtrado.shape[0], "Numero de columnas:", coe2_filtrado.shape[1])

Número de registros para la tabla df_sdem_filtrado: 323124 Numero de columnas: 115
Número de registros para la tabla coe1_filtrado: 323124 Numero de columnas: 191
Número de registros para la tabla coe2_filtrado: 323124 Numero de columnas: 139


In [16]:
cols_coe1 = set(coe1_filtrado.columns)
cols_coe2 = set(coe2_filtrado.columns)
cols_llave = set(llave)

compartidas = cols_coe1 & cols_coe2
exclusivas_coe1 = cols_coe1 - cols_coe2
exclusivas_coe2 = cols_coe2 - cols_coe1
compartidas_sin_llave = compartidas - cols_llave

print("Compartidas",len(compartidas))
print("Exclusivas de COE1",len(exclusivas_coe1))
print("Exclusivas de COE2",len(exclusivas_coe2))
print("Compartidas sin incluir la llave",len(compartidas_sin_llave))

Compartidas 21
Exclusivas de COE1 170
Exclusivas de COE2 118
Compartidas sin incluir la llave 12


Comparemos estas 12 columnas compartidas que no forman parte de la llave. Específicamente, comparemos si comparten el mismo valor en las tablas COE1 y COE2

In [17]:
compartidas_sin_llave = list(compartidas_sin_llave)
comparacion = coe1_filtrado[llave + compartidas_sin_llave].merge(
    coe2_filtrado[llave + compartidas_sin_llave],
    on=llave,
    suffixes = ('_coe1', ('_coe2'))
)

for col in compartidas_sin_llave:
    dif = comparacion[
        comparacion[f'{col}_coe1'] != comparacion[f'{col}_coe2']
    ]
    print(f'{col}: {len(dif)} diferencias')

N_PRO_VIV: 0 diferencias
FAC_MEN: 0 diferencias
N_ENT: 0 diferencias
UR: 0 diferencias
N_INF: 0 diferencias
UPM: 0 diferencias
CVE_MUN: 0 diferencias
EDA: 0 diferencias
CVEGEO: 0 diferencias
FAC_TRI: 0 diferencias
D_SEM: 0 diferencias
PER: 0 diferencias


Las 12 columnas compartidas contienen exactamente los mismos valores en COE1 y COE2 para las 323,124 personas. Por tanto, no conviene quedarse con las dos versiones de las columnas, basta con tener una de ellas.

Verifiquemos cuáles de estas 12 columnas también existen en df_sdem_filtrado

In [18]:
compartidas_sin_llave_dict = {'CVEGEO', 'CVE_MUN', 'D_SEM', 'EDA', 'FAC_MEN', 'FAC_TRI', 'N_ENT', 'N_INF', 'N_PRO_VIV', 'PER', 'UPM', 'UR'}
cols_sdem = set(df_sdem_filtrado.columns)
compartidas_en_3tablas = list(cols_sdem & compartidas_sin_llave_dict)
len(compartidas_en_3tablas)

11

Verifiquemos ahora si los valores de estas 11 columnas coinciden con coe1 y coe2

In [19]:
comparacion_sdem = df_sdem_filtrado[llave + compartidas_en_3tablas].merge(
    coe1_filtrado[llave + compartidas_en_3tablas], #Como ya vimos que los valores de coe1 y coe2 coinciden, podemos usar uno de los dos 
    on=llave,
    suffixes = ('_sdem', ('_coe1'))
)
for col in compartidas_en_3tablas:
    dif = comparacion_sdem[
        comparacion_sdem[f'{col}_sdem'] != comparacion_sdem[f'{col}_coe1']
    ]
    print(f'{col}: {len(dif)} diferencias')

N_PRO_VIV: 0 diferencias
FAC_MEN: 0 diferencias
N_ENT: 0 diferencias
D_SEM: 0 diferencias
CVE_MUN: 0 diferencias
UPM: 0 diferencias
EDA: 323124 diferencias
CVEGEO: 0 diferencias
FAC_TRI: 0 diferencias
UR: 0 diferencias
PER: 0 diferencias


La columna EDA (edad) tiene todos los valores diferentes. La explicación probable es que anteriormente ya habímos convertido EDA a numérico en la tabla sdem

In [20]:
eda_sdem = pd.to_numeric(comparacion_sdem["EDA_sdem"], errors="coerce")
eda_coe1 = pd.to_numeric(comparacion_sdem["EDA_coe1"], errors="coerce")

iguales = (
    eda_sdem.eq(eda_coe1)
    | (eda_sdem.isna() & eda_coe1.isna())
)

print("EDA:", (~iguales).sum(), "diferencias reales")

EDA: 0 diferencias reales


Efectivamente, las edades son las mismas si son convertidas a valores numéricos

Identifiquemos ahora todas las columnas que comparten SDEM y COE1, además de SDEM y COE2.

In [21]:
compartidas_sdem_coe1 = cols_sdem & cols_coe1
compartidas_sdem_coe1_sin_llave = compartidas_sdem_coe1 - cols_llave

compartidas_sdem_coe2 = cols_sdem & cols_coe2
compartidas_sdem_coe2_sin_llave = compartidas_sdem_coe2 - cols_llave

print("Columnas compartidas entre SDEM y COE1 excluyendo la llave:", compartidas_sdem_coe1_sin_llave)
print("Columnas compartidas entre SDEM y COE2 excluyendo la llave:", compartidas_sdem_coe2_sin_llave)

Columnas compartidas entre SDEM y COE1 excluyendo la llave: {'N_PRO_VIV', 'FAC_MEN', 'N_ENT', 'R_DEF', 'UR', 'CVE_MUN', 'UPM', 'EDA', 'CVEGEO', 'FAC_TRI', 'D_SEM', 'PER'}
Columnas compartidas entre SDEM y COE2 excluyendo la llave: {'N_PRO_VIV', 'FAC_MEN', 'N_ENT', 'UR', 'CVE_MUN', 'UPM', 'EDA', 'CVEGEO', 'FAC_TRI', 'D_SEM', 'PER'}


La columna R_DEF está tanto en SDEM como en COE1. Veamos si hay diferencia en sus valores entre estas dos tablas

In [22]:
llave_mas_RDEF = llave + ['R_DEF']

comparacion_RDEF = df_sdem_filtrado[llave_mas_RDEF].merge(
    coe1_filtrado[llave_mas_RDEF],
    on=llave,
    suffixes=("_sdem", "_coe1")
)
iguales = (comparacion_RDEF["R_DEF_sdem"].eq(comparacion_RDEF["R_DEF_coe1"]) | (comparacion_RDEF["R_DEF_sdem"].isna() & comparacion_RDEF["R_DEF_coe1"].isna()))
print("R_DEF:", (~iguales).sum(), "diferencias reales")

R_DEF: 0 diferencias reales


Por lo tanto, ambos valores de las tablas corresponden con cada registro, por lo que solo nos quedaremos con una de ellas.

Veamos ahora cuáles columnas son exclusivas de cada tabla y su tipo de datos

In [23]:
# Para recordar, vuelvo a transformar en conjuntos los nombres de las columnas

cols_sdem = set(df_sdem_filtrado.columns)
cols_coe1 = set(coe1_filtrado.columns)
cols_coe2 = set(coe2_filtrado.columns)
cols_llave = set(llave)

exclusivas_sdem = cols_sdem - cols_llave - cols_coe1 - cols_coe2
exclusivas_coe1 = cols_coe1 - cols_llave - cols_sdem - cols_coe2
exclusivas_coe2 = cols_coe2 - cols_llave - cols_sdem - cols_coe1

print("Exclusivas de SDEM:", len(exclusivas_sdem))
print(df_sdem_filtrado.dtypes.value_counts())

print("Exclusivas de COE1:", len(exclusivas_coe1))
print(coe1_filtrado.dtypes.value_counts())

print("Exclusivas de COE2:", len(exclusivas_coe2))
print(coe2_filtrado.dtypes.value_counts())

Exclusivas de SDEM: 94
int64      63
object     51
float64     1
Name: count, dtype: int64
Exclusivas de COE1: 169
object    188
int64       3
Name: count, dtype: int64
Exclusivas de COE2: 118
object    136
int64       3
Name: count, dtype: int64


Comprobemos, para cada columna, el número de registros con cadena vacia

In [24]:
def n_cols_vacias(dataframe):
    col_vacia = 0
    cols = list(dataframe.columns)
    for col in cols:
        cad_vacia = dataframe[col].astype(str).str.strip().eq("").sum()
        if cad_vacia > 0:
            col_vacia += 1
    print("numero de columnas con al menos una cadena vacia: ", col_vacia)

print("Para SDEM,",n_cols_vacias(df_sdem_filtrado))
print("Para COE1,",n_cols_vacias(coe1_filtrado))
print("Para COE2,",n_cols_vacias(coe2_filtrado))

numero de columnas con al menos una cadena vacia:  17
Para SDEM, None
numero de columnas con al menos una cadena vacia:  169
Para COE1, None
numero de columnas con al menos una cadena vacia:  118
Para COE2, None


Se puede observar que COE1 contiene 169 columnas con al menos una cadena vacía, así como 118 para el caso de COE2. Este número coincide con la cantidad de columnas exclusivas de cada tabla.

Identifiquemos cuántas de estas columnas caen en los siguientes intervalos
+ 0 %
+ más de 0 % y hasta 25 %
+ más de 25 % y hasta 50 %
+ más de 50 % y hasta 75 %
+ más de 75 %

In [25]:
def resumen_cadenas_vacias(dataframe):
    total = len(dataframe)

    resumen = pd.DataFrame({
        "columna": dataframe.columns,
        "num_vacios": [
            dataframe[col].astype(str).str.strip().eq("").sum()
            for col in dataframe.columns
        ]
    })

    resumen["porcentaje_vacios"] = (
        resumen["num_vacios"] / total * 100
    )

    resumen["intervalo"] = pd.cut(
        resumen["porcentaje_vacios"],
        bins=[-0.01, 0, 25, 50, 75, 100],
        labels=[
            "0 %",
            "> 0 % y ≤ 25 %",
            "> 25 % y ≤ 50 %",
            "> 50 % y ≤ 75 %",
            "> 75 %"
        ]
    )

    conteo_intervalos = (
        resumen["intervalo"]
        .value_counts(sort=False)
    )

    return resumen, conteo_intervalos

# Para SDEM
resumen_sdem, intervalos_sdem = resumen_cadenas_vacias(df_sdem_filtrado)
print("SDEM:\n", intervalos_sdem)

# Para COE1
resumen_coe1, intervalos_coe1 = resumen_cadenas_vacias(coe1_filtrado)
print("COE1:\n",intervalos_coe1)

# Para COE2
resumen_coe2, intervalos_coe2 = resumen_cadenas_vacias(coe2_filtrado)
print("COE2:\n",intervalos_coe2)

SDEM:
 intervalo
0 %                98
> 0 % y ≤ 25 %      5
> 25 % y ≤ 50 %     1
> 50 % y ≤ 75 %     3
> 75 %              8
Name: count, dtype: int64
COE1:
 intervalo
0 %                 22
> 0 % y ≤ 25 %       1
> 25 % y ≤ 50 %     27
> 50 % y ≤ 75 %     17
> 75 %             124
Name: count, dtype: int64
COE2:
 intervalo
0 %                21
> 0 % y ≤ 25 %      6
> 25 % y ≤ 50 %    12
> 50 % y ≤ 75 %     6
> 75 %             94
Name: count, dtype: int64


Veamos cuáles son las columnas con 100% de cadenas vacias

In [26]:
# Para SDEM
resumen_sdem[resumen_sdem["porcentaje_vacios"]==100].iloc[:,:-1]

,columna,num_vacios,porcentaje_vacios
1,CVE_LOC,323124,100.0
43,CS_AD_MOT,323124,100.0
44,CS_P21_DES,323124,100.0
45,CS_AD_DES,323124,100.0


In [27]:
# Para coe1
resumen_coe1[resumen_coe1["porcentaje_vacios"]==100].iloc[:,:-1]

,columna,num_vacios,porcentaje_vacios
113,P4_1,323124,100.0
114,P4_2,323124,100.0
117,P4A_1,323124,100.0


In [28]:
# Para coe2
resumen_coe2[resumen_coe2["porcentaje_vacios"]==100].iloc[:,:-1]

,columna,num_vacios,porcentaje_vacios


Estudiando los campos con 100% de vacios en la documentación, podemos clasificar cada campo de la siguiente manera

+ CVE_LOC: omitida por confidencialidad; no disponible en los microdatos públicos.
+ CS_AD_MOT, CS_AD_DES, CS_P21_DES: vacíos estructurales porque su universo corresponde a C_RES = "2", población ya excluida.
+ P4_1, P4_2, P4A_1: campos internos o exclusivos del sistema, sin utilidad analítica.

**Conclusión metodológica:** se revisaron manualmente las variables con 100 % de valores vacíos para identificar su causa. Se distinguieron campos omitidos por confidencialidad, variables internas del sistema y vacíos estructurales derivados de filtros de población o saltos del cuestionario.

La revisión individual se limitará posteriormente a la variable objetivo, los predictores candidatos y los campos que presenten patrones inconsistentes con su universo de aplicación.

## Definición de la variable objetivo
Según la documentación oficial, el campo que clasifica si un entrevistado tiene empleo formal o informal es EMP_PPAL, con valores 1 y 2. No confundir con TUE_PPAL que clasifica el tipo de unidad económica. Una persona puede tener empleo informal fuera del sector informal, por ejemplo, trabajando sin seguridad social en una empresa formal. Nosotros utilizaremos **EMP_PPAL**
Estudiemos esta variable

In [29]:
df_sdem_filtrado["EMP_PPAL"].value_counts()

EMP_PPAL
0    135567
1     97079
2     90478
Name: count, dtype: int64

Sin embargo, la documentación no toma a 0 como valor válido. Estudiemos su causa

In [30]:
len(df_sdem_filtrado[(df_sdem_filtrado["CLASE2"]==2) | (df_sdem_filtrado["CLASE2"]==3) | (df_sdem_filtrado["CLASE2"]==4)])

135567

También tenemos 135567 que coinciden con EMP_PPAL=0. Veamos si corresponden a los mismos registros

In [31]:
pd.crosstab(
    df_sdem_filtrado["CLASE2"],
    df_sdem_filtrado["EMP_PPAL"],
    rownames=["CLASE2"],
    colnames=["EMP_PPAL"],
    margins=True
)

EMP_PPAL,0,1,2,All
CLASE2,,,,
1,0,97079,90478,187557
2,4940,0,0,4940
3,14521,0,0,14521
4,116106,0,0,116106
All,135567,97079,90478,323124


La tabla cruzada confirma que `EMP_PPAL = 0` aparece exclusivamente en personas con `CLASE2 ∈ {2, 3, 4}`. Por tanto, se interpreta operativamente como población fuera del universo de clasificación del empleo principal.

La población analítica del proyecto se restringirá a personas ocupadas, identificadas mediante `CLASE2 = 1`. Dentro de esta población, `EMP_PPAL` distingue entre empleo informal (`1`) y empleo formal (`2`).

In [32]:
df_ocupados = df_sdem_filtrado[df_sdem_filtrado["CLASE2"].eq(1)]
len(df_ocupados)

187557

La población ocupada contiene 187,557 registros.

Comprobemos entonces que 
+ hay exactamente 187,557 registros;
+ CLASE2 solo contiene 1;
+ EMP_PPAL solo contiene 1 y 2;
+ No existen valores vacíos en EMP_PPAL.

In [33]:
print(df_ocupados["CLASE2"].value_counts())

print(df_ocupados["EMP_PPAL"].value_counts())

print("Valores vacios",df_ocupados["EMP_PPAL"].isna().sum())

print("Valores con cadena vacia",df_ocupados["EMP_PPAL"].astype(str).str.strip().eq("").sum())

CLASE2
1    187557
Name: count, dtype: int64
EMP_PPAL
1    97079
2    90478
Name: count, dtype: int64
Valores vacios 0
Valores con cadena vacia 0


La variable objetivo presenta una distribución muestral prácticamente equilibrada: 51.76 % empleo informal y 48.24 % empleo formal. No se observa un problema de desbalance de clases que justifique técnicas de remuestreo en esta etapa. No obstante, la prevalencia poblacional deberá estimarse posteriormente utilizando el factor de expansión trimestral FAC_TRI.

In [34]:
distribucion_ponderada = (
    df_ocupados
    .groupby("EMP_PPAL")["FAC_TRI"]
    .sum()
)

porcentajes_ponderados = (
    distribucion_ponderada
    / distribucion_ponderada.sum()
    * 100
)

print(distribucion_ponderada)
print(porcentajes_ponderados)

EMP_PPAL
1    32614387
2    26913198
Name: FAC_TRI, dtype: int64
EMP_PPAL
1    54.788695
2    45.211305
Name: FAC_TRI, dtype: float64


Al aplicar el factor de expansión trimestral `FAC_TRI`, la muestra de 187,557 personas ocupadas representa una población estimada de 59,527,585 personas.

De este total:

* 32,614,387 corresponden a empleo informal, equivalente al 54.79 %.
* 26,913,198 corresponden a empleo formal, equivalente al 45.21 %.

La distribución ponderada muestra una mayor prevalencia de empleo informal que la distribución no ponderada de la muestra, donde representaba el 51.76 %.

Ya hemos determinado nuestra variable objetivo EMP_PAL. Como es una variable construida por INEGI, algunas coluymnas pueden formar parte directa de su cálculo o describir casi exactamente la formalidad laboral. Usarlas como predictores harían que el modelo "reconstruya" la etiqueta en lugar de anticiparla. Según el documento "Reconstrucción de variables" (https://www.inegi.org.mx/contenidos/programas/enoe/15ymas/doc/recons_var_15ymas.pdf), se determina la informalidad en función de las variables `TUE2`, `POS_OCU`, `RAMA`, `REMUNE2C` y `SEG_SOC`. Se consideran predictores con fuga directa y no deberán utilizarse en el modelo. Verifiquemos cuáles de estas cinco variables están presentes en df_ocupados

In [35]:
df_ocupados[["TUE2", "POS_OCU", "RAMA", "REMUNE2C", "SEG_SOC"]].head(1)

,TUE2,POS_OCU,RAMA,REMUNE2C,SEG_SOC
0,3,1,4,1,1


Para documentar el riesgo de fuga de información, se reconstruyó la dependencia entre `EMP_PPAL`, las variables derivadas que intervienen directamente en su definición y las preguntas originales que las generan.

| Variable derivada | Variables fuente identificadas                                                                                                                                    | Tipo de fuga                                     |
| ----------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------ |
| `TUE2`            | `P3`, `P3B`, `P3C1`, `P3C2`, `P3C3`, `P3C4`, `P3C9`, `P3D`, `P3G1_1`, `P3H`, `P3Q`, `P4`, `P4A`, `P4B`, `P4C`, `P4D1`, `P4D2`, `P4D3`, `P4E`, `P4G`, `P4I`, `P6D` | `TUE2`: directa. Variables fuente: indirecta.    |
| `POS_OCU`         | `P3H`, `P3D`, `P3G1_1`                                                                                                                                            | `POS_OCU`: directa. Variables fuente: indirecta. |
| `RAMA`            | `P4A`                                                                                                                                                             | `RAMA`: directa. `P4A`: indirecta.               |
| `REMUNE2C`        | `P3H`                                                                                                                                                             | `REMUNE2C`: directa. `P3H`: indirecta.           |
| `SEG_SOC`         | `P6D`, `P3L1`, `P3L2`, `P3L3`, `P3L9`, `P3M1`–`P3M7`, `P3M9`                                                                                                      | `SEG_SOC`: directa. Variables fuente: indirecta. |

Las variables derivadas se consideran fuga directa porque participan explícitamente en las reglas de construcción de `EMP_PPAL`. Sus variables fuente presentan fuga indirecta, ya que contienen la información utilizada para generar dichas clasificaciones. En consecuencia, todas deberán quedar excluidas del conjunto de predictores, salvo que posteriormente se justifique un escenario de modelado alternativo.

Veamos en dónde están todas estas variables

In [36]:
variables_fuga = ["TUE2", "POS_OCU", "RAMA", "REMUNE2C", "SEG_SOC","P3", "P3B", "P3C1", "P3C2", "P3C3", "P3C4", "P3C9", "P3D", "P3G1_1", "P3H", "P3Q", "P4", "P4A", "P4B", "P4C", "P4D1","P4D2", 
                     "P4D3", "P4E", "P4G", "P4I", "P6D","P3L1", "P3L2", "P3L3", "P3L9", "P3M1", "P3M2","P3M3","P3M4","P3M5","P3M6","P3M7","P3M9"]

resultado = []

for variable in variables_fuga:
    resultado.append({
        "Variable": variable,
        "SDEM": variable in df_ocupados.columns,
        "COE1": variable in coe1_filtrado.columns,
        "COE2": variable in coe2_filtrado.columns,
    })

df_resultado = pd.DataFrame(resultado)
df_resultado

,Variable,SDEM,COE1,COE2
0,TUE2,True,False,False
1,POS_OCU,True,False,False
2,RAMA,True,False,False
3,REMUNE2C,True,False,False
4,SEG_SOC,True,False,False
5,P3,False,True,False
6,P3B,False,True,False
7,P3C1,False,True,False
8,P3C2,False,True,False
9,P3C3,False,True,False


Con lo anterior, observamos que
+ Las cinco variables de fuga directa están en SDEM: TUE2, POS_OCU, RAMA, REMUNE2C y SEG_SOC.
+ La mayoría de las variables fuente de fuga indirecta están en COE1.
+ P6D está en COE2.
+ No hay variables de esta lista ausentes en las tres tablas.

las 5 directas deben excluirse, así como las 34 variables indirectas.

Antes de unir, definamos qué columnas retirar de COE1 y COE2 por estar ya presentes en SDEM y construyamos las columnas que se incorporarán

Ahora construyamos las columnas que realmente se incorporarán

In [37]:
comunes_sdem_coe1 = set(df_ocupados.columns) & set(coe1_filtrado.columns)
comunes_sdem_coe2 = set(df_ocupados.columns) & set(coe2_filtrado.columns)
comunes_coe1_coe2 = set(coe1_filtrado.columns) & set(coe2_filtrado.columns)

columnas_coe1_agregar = [
    col for col in coe1_filtrado.columns
    if col not in comunes_sdem_coe1
]

columnas_coe2_agregar = [
    col for col in coe2_filtrado.columns
    if col not in comunes_sdem_coe2
    and col != "N_INF"
]

columnas_coe1_merge = llave + columnas_coe1_agregar
columnas_coe2_merge = llave + columnas_coe2_agregar

print(len(columnas_coe1_merge) == len(set(columnas_coe1_merge)))
print(len(columnas_coe2_merge) == len(set(columnas_coe2_merge)))

True
True


# Union de las tablas

Realicemos la unión entre SDEM y COE1 y validemoslo

In [38]:
df_sdem_coe1 = df_ocupados.merge(
    coe1_filtrado[columnas_coe1_merge],
    on = llave,
    how = 'left',
    validate = 'one_to_one'
)

print(df_sdem_coe1.shape)
print(len(df_sdem_coe1) == len(df_ocupados))
print(df_sdem_coe1.duplicated(subset=llave).sum())

(187557, 285)
True
0


Es correcto. Ahora, unamoslo con COE2

In [39]:
df_final = df_sdem_coe1.merge(
    coe2_filtrado[columnas_coe2_merge],
    on = llave,
    how = 'left',
    validate = 'one_to_one'
)

print(df_final.shape)
print(len(df_final) == len(df_ocupados))
print(df_final.duplicated(subset=llave).sum())

(187557, 403)
True
0


Tenemos entonces 403 variables, las cuales deberemos estudiar para decidir cuáles de ellas serán utiles para el modelo. 

Primero, eliminemos las variables fuga

In [40]:
df_modelado = df_final.drop(columns=variables_fuga)

set(variables_fuga) & set(df_modelado.columns)

set()

Eliminemos ahora las columnas constantes

In [41]:
columnas_constantes = [col for col in df_modelado.columns
                       if df_modelado[col].nunique(dropna=False)<=1]
len(columnas_constantes)

55

Estas columnas no aportan información predictiva ya que no cambia entre observaciones.

In [42]:
df_modelado = df_modelado.drop(columns=columnas_constantes)

print(df_modelado.shape)
print(df_modelado.columns.intersection(columnas_constantes).tolist())

(187557, 309)
[]


Separemos ahora las variables que deben conservase solo para trazabilidad o evaluación, pero no como predictores, entre ellas:
+ llave compuesta
+ EMP_PPAL (objetivo)
+ FAC_TRI y FAC_MEN

Primero, revisemos cuáles de estas siguen presentes

In [43]:
columnas_no_predictoras = llave + ["EMP_PPAL", "FAC_TRI", "FAC_MEN"]

[col for col in columnas_no_predictoras if col in df_modelado.columns]

['TIPO',
 'MES_CAL',
 'CD_A',
 'CVE_ENT',
 'CON',
 'V_SEL',
 'N_HOG',
 'H_MUD',
 'N_REN',
 'EMP_PPAL',
 'FAC_TRI',
 'FAC_MEN']

Por el momento no las eliminaremos, haremos su separación después

Hagamos una tabla resumen con porcentaje de vacios

In [45]:
resumen_columnas = pd.DataFrame({
    "Variable": df_modelado.columns,
    "Tipo": df_modelado.dtypes.astype(str).values,
    "Unicos": [df_modelado[col].nunique(dropna=False) for col in df_modelado.columns],
    "Vacios": [(df_modelado[col] == "").sum() if df_modelado[col].dtype == "object" else df_modelado[col].isna().sum()
               for col in df_modelado.columns]
})

resumen_columnas["Porcentaje_vacios"] = (
    resumen_columnas["Vacios"] / len(df_modelado) * 100
)

Veamos los 15 primeros puestos con porcentajes de vacios de mayor a menor

In [46]:
resumen_columnas.sort_values("Porcentaje_vacios", ascending=False).head(15)

,Variable,Tipo,Unicos,Vacios,Porcentaje_vacios
230,P7G2,object,2,187550,99.996268
268,P9N2,object,2,187546,99.994135
263,P9M2,object,2,187546,99.994135
270,P9N4,object,2,187542,99.992002
113,P3G4_2,object,4,187537,99.989337
112,P3G4_1,object,2,187537,99.989337
264,P9M3,object,2,187523,99.981872
190,P5G99,object,2,187520,99.980273
271,P9N5,object,2,187517,99.978673
273,P9N9,object,2,187510,99.974941


Con esta depuración se obtiene un dataframe de modelado inicial con 187,557 personas ocupadas y 309 columnas, después de eliminar variables con fuga de información y columnas constantes. Las variables de trazabilidad, objetivo y ponderación se conservan separadas conceptualmente, pero no deberán entrar directamente como predictores. A partir de este punto, la selección de variables se orientará hacia un modelo operativo reducido, basado en variables interpretables y disponibles en un formulario breve.


## Selección de variables para modelo operativo reducido

A partir de este punto, la selección de predictores se orienta hacia un escenario operativo. El objetivo no es replicar la clasificación de informalidad contenida en la ENOE usando todo el cuestionario, sino construir un modelo de tamizaje que estime la probabilidad de empleo informal con un conjunto reducido de variables que podrían ser conocidas o preguntadas por un analista mediante un formulario breve.

Por ello, las variables candidatas deberán cumplir cuatro criterios:

1. estar disponibles antes de tomar una decisión de política pública;
2. ser interpretables para un analista no especializado;
3. no formar parte directa ni indirecta de la definición de `EMP_PPAL`;
4. ser razonablemente capturables en un formulario de aproximadamente 15 a 20 preguntas.

Bajo este enfoque, una variable puede ser predictiva en la ENOE y aun así excluirse si no es operativamente útil o si requiere información demasiado específica del cuestionario.


Se proponen los siguientes grupos de variables
| Grupo                                 | Tipo de información                   | Ejemplos esperados                                                      |
| ------------------------------------- | ------------------------------------- | ----------------------------------------------------------------------- |
| Sociodemográficas                     | Perfil básico de la persona           | edad, sexo, escolaridad, estado civil                                   |
| Ubicación                             | Contexto territorial                  | entidad, urbano/rural, municipio si se decide usarlo                    |
| Características generales del trabajo | Información laboral preguntable       | ocupación general, sector, tamaño del establecimiento, lugar de trabajo |
| Trayectoria/estabilidad laboral       | Condiciones generales no definitorias | antigüedad, estacionalidad, búsqueda de otro empleo                     |


La selección inicial de variables se realizó identificando un conjunto reducido de predictores que pudieran ser conocidos o preguntados por un analista mediante un formulario breve.

Para ello, se priorizaron variables con cuatro características: disponibilidad práctica al momento de la consulta, interpretación clara para usuarios no especializados, ausencia de fuga directa o indirecta respecto a `EMP_PPAL`, y relevancia sustantiva para caracterizar el perfil sociodemográfico, territorial y laboral de la persona ocupada.

También se descartaron variables demasiado específicas del cuestionario, variables condicionadas con muchos blancos estructurales, campos derivados de dimensiones usadas en la construcción de la informalidad y variables que, aunque predictivas, podrían convertir el modelo en una reconstrucción indirecta de la clasificación oficial.

La primera versión de variables son las siguientes
| **Variable** | **Grupo** | **Motivo** | **Tratamiento futuro** |
| --- | --- | --- | --- |
| ``SEX`` | Sociodemográficas | Variable básica del perfil individual, disponible y fácil de capturar. | Codificar como categórica nominal. |
| ``EDA`` | Sociodemográficas | La edad es central para caracterizar perfiles laborales. | Usar como numérica o agrupar por rangos de edad. |
| ``CS_P13_1`` | Sociodemográficas | Resume el nivel escolar aprobado, relevante para empleabilidad e inserción laboral. | Tratar como categórica ordinal. Revisar códigos especiales. |
| ``E_CON`` | Sociodemográficas | El estado conyugal puede asociarse con responsabilidades económicas y laborales. | Codificar como categórica nominal. |
| ``PAR_C`` | Sociodemográficas / hogar | Captura la posición de la persona dentro del hogar, sin relación directa con informalidad. | Reagrupar en jefe/a, pareja, hijo/a, otros familiares, no familiares y no especificado. |
| ``CVE_ENT`` | Ubicación | Permite capturar diferencias territoriales estatales, útil para política pública. | Codificar como categórica nominal. |
| ``T_LOC_TRI`` | Ubicación | Resume el tamaño de localidad en categorías interpretables y operativas. | Usar como categórica ordinal. |
| ``DUR_EST`` | Características generales del trabajo | Representa la duración de la jornada en rangos simples y preguntables. | Codificar como categórica ordinal. |
| ``ING7C`` | Características generales del trabajo | Resume el ingreso laboral en rangos, evitando usar ingreso exacto. | Tratar como categórica ordinal; conservar “no recibe ingresos” y “no especificado”. |
| ``P3R_ANIO`` | Trayectoria / estabilidad laboral | Aproxima la antigüedad en el trabajo actual con buena cobertura. | Transformar a antigüedad laboral aproximada; tratar ``9999`` como “no sabe”. |
| ``P3A`` | Características generales del trabajo | Indica si la persona tiene jefe o superior; simple, operativo y con cobertura completa. | Codificar como binaria; documentar cercanía conceptual con subordinación laboral. |
| ``BUSCAR5C`` | Trayectoria / estabilidad laboral | Resume si la persona buscó otro empleo o cambiarse de trabajo, señal de estabilidad o insatisfacción laboral. | Codificar como categórica nominal; mantener “no especificado” como categoría. |

Guardemos un csv para seguir trabajando en la notebook 03_eda_preparacion_modelo.ipynb

In [87]:
variables_modelo = [
    "SEX",
    "EDA",
    "CS_P13_1",
    "E_CON",
    "PAR_C",
    "CVE_ENT",
    "T_LOC_TRI",
    "DUR_EST",
    "ING7C",
    "P3R_ANIO",
    "P3A",
    "BUSCAR5C"
]
df_operativo_v1 = df_modelado[columnas_no_predictoras + variables_modelo].copy()

df_modelado.to_csv("/home/jeffrey/Documentos/Proyectos/enoe_informalidad/data/processed/df_modelado_auditado.csv", index=False)
df_operativo_v1.to_csv("/home/jeffrey/Documentos/Proyectos/enoe_informalidad/data/processed/df_operativo_v1.csv", index=False)

Con esta sección se cierra la fase de comprensión y auditoría inicial de datos. Se construyó el universo analítico de personas ocupadas, se validó la cobertura entre SDEM, COE1 y COE2, se definió la variable objetivo `EMP_PPAL`, se eliminaron variables con fuga de información y columnas constantes, y se seleccionó un conjunto inicial de variables para un modelo operativo reducido.

A partir de este punto, el proyecto continúa con una nueva notebook orientada al tratamiento de variables, análisis exploratorio del conjunto operativo, recodificación, construcción de variables derivadas y preparación de datos para modelado.
